In [3]:
import numpy as np
from PIL import Image
from typing import Tuple


def load_image(path: str) -> np.ndarray:
    """Загрузка цветного изображения в RGB."""
    img = Image.open(path).convert("RGB")
    return np.array(img, dtype=np.float32) / 255.0


def save_image(image: np.ndarray, path: str) -> None:
    """Сохранение цветного изображения."""
    img = Image.fromarray((image * 255).astype(np.uint8))
    img.save(path)


def create_mask(image: np.ndarray, hole_ratio: float) -> np.ndarray:
    """Создание маски с дырами (1 — битый пиксель, 0 — известный пиксель)."""
    h, w, _ = image.shape
    mask = np.zeros((h, w), dtype=np.float32)
    hole_pixels = int(hole_ratio * h * w)
    coords = np.random.choice(h * w, hole_pixels, replace=False)
    mask.flat[coords] = 1
    return mask


def restricted_laplacian(u_channel: np.ndarray, mask: np.ndarray) -> np.ndarray:
    """Лапласиан, который обновляет только пиксели в дырах."""
    h, w = u_channel.shape
    laplacian_u = np.zeros_like(u_channel)

    # Создаем расширенную маску для граничных пикселей
    expanded_mask = np.zeros_like(mask)
    expanded_mask[1:-1, 1:-1] = mask[1:-1, 1:-1]

    # Вычисляем лапласиан только для битых пикселей и их соседей
    for i in range(1, h - 1):
        for j in range(1, w - 1):
            if mask[i, j] == 1 or (
                mask[i - 1, j] == 1
                or mask[i + 1, j] == 1
                or mask[i, j - 1] == 1
                or mask[i, j + 1] == 1
            ):
                laplacian_u[i, j] = (
                    u_channel[i + 1, j]
                    + u_channel[i - 1, j]
                    + u_channel[i, j + 1]
                    + u_channel[i, j - 1]
                    - 4 * u_channel[i, j]
                )

    return laplacian_u * expanded_mask


def conjugate_gradient_inpainting_channel(
    channel: np.ndarray, mask: np.ndarray, max_iter: int = 100, tol: float = 1e-4
) -> np.ndarray:
    """Inpainting (восстановление пикселей) для одного канала."""
    u = channel.copy()
    h, w = u.shape

    # Инициализация только в битых пикселях
    b = -restricted_laplacian(u, mask)  # Правая часть уравнения
    p = b.copy()  # Начальное направление поиска
    r = b.copy()  # Начальная невязка
    rs_old = np.sum(r**2)

    for _ in range(max_iter):
        # Применяем лапласиан только к активным пикселям
        Ap = restricted_laplacian(p, mask)

        # Вычисляем шаг только для битых пикселей
        active_pixels = mask == 1
        alpha = rs_old / np.sum(p[active_pixels] * Ap[active_pixels])

        # Обновляем битые пиксели
        update = alpha * p
        u[active_pixels] += update[active_pixels]
        r[active_pixels] -= (alpha * Ap)[active_pixels]

        # Проверка сходимости
        rs_new = np.sum(r[active_pixels] ** 2)
        if np.sqrt(rs_new) < tol:
            break

        # Обновление направления
        beta = rs_new / rs_old
        p[active_pixels] = r[active_pixels] + beta * p[active_pixels]
        rs_old = rs_new

    return u


def conjugate_gradient_inpainting(
    image: np.ndarray, mask: np.ndarray, max_iter: int = 100, tol: float = 1e-4
) -> np.ndarray:
    """Inpainting (восстановление пикселей) для цветного изображения с сохранением известных пикселей."""
    # Сохраняем оригинальные пиксели
    original_pixels = image * (1 - mask[..., np.newaxis])

    # Разделение на каналы
    r_channel = image[:, :, 0]
    g_channel = image[:, :, 1]
    b_channel = image[:, :, 2]

    # Восстановление каждого канала
    r_restored = conjugate_gradient_inpainting_channel(r_channel, mask, max_iter, tol)
    g_restored = conjugate_gradient_inpainting_channel(g_channel, mask, max_iter, tol)
    b_restored = conjugate_gradient_inpainting_channel(b_channel, mask, max_iter, tol)

    # Объединение каналов и восстановление оригинальных пикселей
    restored_image = np.stack([r_restored, g_restored, b_restored], axis=-1)
    restored_image = np.where(
        mask[..., np.newaxis] == 1, restored_image, original_pixels
    )

    return restored_image

In [4]:
# Загрузка изображения и создание маски
image = load_image("origins/friren.jpg")

# Имитация повреждения
mask = create_mask(image, hole_ratio=0.6)
damaged_image = image * (1 - mask[..., np.newaxis])

# Восстановление (только битые пиксели)
restored = conjugate_gradient_inpainting(damaged_image, mask)

# Сохранение результатов
save_image(damaged_image, "results/task_1/damaged.jpg")
save_image(restored, "results/task_1/restored.jpg")